In [ ]:
# ============================================================
# 1. INSTALLATION AND IMPORTS
# ============================================================

!pip install -q tabpfn

import os
import gc
import numpy as np
import pandas as pd
import torch
import joblib

from google.colab import userdata

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

from tabpfn import TabPFNClassifier


# ============================================================
# 2. CONFIGURATION
# ============================================================

drive_path = '/content/drive/MyDrive/datasets_gaw'

merged_path = os.path.join(
    drive_path,
    'firms_era5_nearest_neighbor.csv'
)

RANDOM_STATE = 42

TARGET_COLUMN = 'wildfire'


# ============================================================
# 3. TABPFN AUTHENTICATION
# ============================================================

print("Loading TabPFN token...")

token = userdata.get('TABPFN_TOKEN')

if not token:
    raise ValueError(
        "TABPFN_TOKEN not found in Colab Secrets."
    )

os.environ['TABPFN_TOKEN'] = token

print("TabPFN token retrieved successfully.")


# ============================================================
# 4. LOAD FINAL MERGED DATASET
# ============================================================

print("\nLoading final merged dataset...")

if not os.path.exists(merged_path):
    raise FileNotFoundError(merged_path)

data = pd.read_csv(merged_path)

print("Dataset shape:", data.shape)

data['acq_date'] = pd.to_datetime(
    data['acq_date'],
    errors='coerce'
)

data = data.dropna(
    subset=['acq_date', 'detection_count']
).copy()

data['year'] = data['acq_date'].dt.year



In [ ]:

# ============================================================
# 5. CREATE TARGET VARIABLE
# ============================================================

# 1 = more than one FIRMS detection
# 0 = one FIRMS detection

data[TARGET_COLUMN] = (
    data['detection_count'] > 1
).astype(int)

print("\nTarget distribution:")

print(data[TARGET_COLUMN].value_counts())

print("\nTarget proportions:")

print(
    data[TARGET_COLUMN].value_counts(
        normalize=True
    )
)


# ============================================================
# 6. SELECT WEATHER FEATURES
# ============================================================

weather_cols = [
    'swvl1', 'swvl2', 'swvl3',
    'lai_lv', 'lai_hv',
    'sp', 'slhf',
    'u10', 'v10',
    't2m', 'd2m',
    'ssrd', 'ssr', 'str',
    'tp', 'skt', 'pev',
    'sde', 'evatc', 'evabs', 'evavt',
    'snowc', 'cvh', 'cvl',
    'z', 'lsm'
]

feature_cols = [
    col for col in weather_cols
    if col in data.columns
]

if len(feature_cols) == 0:
    raise ValueError("No ERA5 weather features found.")

print("\nNumber of weather features:", len(feature_cols))

print("Features:", feature_cols)

# Convert weather columns to numeric
for col in feature_cols:
    data[col] = pd.to_numeric(
        data[col],
        errors='coerce'
    )

# Replace infinite values
data[feature_cols] = data[
    feature_cols
].replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove rows where every weather feature is missing
data = data.dropna(
    subset=feature_cols,
    how='all'
).copy()



In [ ]:

# ============================================================
# 7. TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

print("\nCreating temporal splits...")

train_df = data[
    data['year'].between(2018, 2022)
].copy()

val_df = data[
    data['year'] == 2023
].copy()

test_df = data[
    data['year'] == 2024
].copy()

if train_df.empty or val_df.empty or test_df.empty:
    raise ValueError(
        "One or more temporal splits are empty."
    )

print("\n===================================")
print("TEMPORAL DATASET SPLITS")
print("===================================")

print("Training (2018-2022):", len(train_df))
print("Validation (2023):", len(val_df))
print("Testing (2024):", len(test_df))

print("\nTraining years:")
print(sorted(train_df['year'].unique()))

print("\nValidation years:")
print(sorted(val_df['year'].unique()))

print("\nTest years:")
print(sorted(test_df['year'].unique()))


# ============================================================
# 8. CHECK CLASS DISTRIBUTION
# ============================================================

print("\nTraining class distribution:")
print(train_df[TARGET_COLUMN].value_counts())

print("\nValidation class distribution:")
print(val_df[TARGET_COLUMN].value_counts())

print("\nTest class distribution:")
print(test_df[TARGET_COLUMN].value_counts())

for name, df in [
    ('Training', train_df),
    ('Validation', val_df),
    ('Test', test_df)
]:
    if df[TARGET_COLUMN].nunique() != 2:
        raise ValueError(
            f"{name} dataset does not contain both classes."
        )



In [ ]:

# ============================================================
# 9. PREPARE X AND Y
# ============================================================

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[TARGET_COLUMN].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET_COLUMN].copy()

print("\nOriginal training samples:", len(X_train))


# ============================================================
# 10. USE ALL TRAINING DATA (SUBSAMPLING REMOVED)
# ============================================================

print("\nUsing all available training samples:", len(X_train))

print("Training class distribution:")
print(y_train.value_counts())


# ============================================================
# 11. IMPUTATION
# ============================================================

print("\nHandling missing values...")

imputer = SimpleImputer(
    strategy='median'
)

X_train_imputed = imputer.fit_transform(
    X_train
)

X_val_imputed = imputer.transform(
    X_val
)

X_test_imputed = imputer.transform(
    X_test
)


# ============================================================
# 12. FEATURE SCALING
# ============================================================

print("\nScaling features...")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_imputed
)

X_val_scaled = scaler.transform(
    X_val_imputed
)

X_test_scaled = scaler.transform(
    X_test_imputed
)

print("Feature preprocessing completed.")

print("Training matrix:", X_train_scaled.shape)
print("Validation matrix:", X_val_scaled.shape)
print("Test matrix:", X_test_scaled.shape)


In [ ]:
############# run from here for the model training




# ============================================================
# 13. AUTHENTICATE AND INITIALIZE TABPFN
# ============================================================

import os
import torch
import numpy as np

from google.colab import userdata
from tabpfn import TabPFNClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

print("\n===================================")
print("TABPFN MODEL INITIALIZATION")
print("===================================")

# ------------------------------------------------------------
# 1. Load TabPFN API key from Google Colab Secrets
# ------------------------------------------------------------

try:
    tabpfn_token = userdata.get("TABPFN_TOKEN")

    if not tabpfn_token:
        raise ValueError("TABPFN_TOKEN is empty.")

    os.environ["TABPFN_TOKEN"] = tabpfn_token

    print("TabPFN authentication configured successfully.")

except Exception as e:
    raise RuntimeError(
        "Unable to load TABPFN_TOKEN. "
        "Check that you added TABPFN_TOKEN to Google Colab Secrets "
        "and enabled notebook access."
    ) from e


# ------------------------------------------------------------
# 2. Detect available hardware
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nDevice:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Using CPU.")


# ------------------------------------------------------------
# 3. Validate training data
# ------------------------------------------------------------

print("\nChecking training data...")

print("X_train shape:", X_train_scaled.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val_scaled.shape)
print("y_val shape:", y_val.shape)

print("Training classes:", np.unique(y_train))
print("Validation classes:", np.unique(y_val))

if len(np.unique(y_train)) < 2:
    raise ValueError(
        "Training data must contain both classes (0 and 1)."
    )

if len(np.unique(y_val)) < 2:
    raise ValueError(
        "Validation data must contain both classes (0 and 1) "
        "to calculate ROC-AUC."
    )


In [ ]:

# ============================================================
# 14. TRAIN MODEL
# ============================================================

print("\nTraining TabPFN...")

classifier = TabPFNClassifier(device=device)

classifier.fit(
    X_train_scaled,
    y_train
)

print("\nTabPFN training completed!")


# ============================================================
# 15. VALIDATION EVALUATION
# ============================================================

print("\n===================================")
print("VALIDATION RESULTS (2023)")
print("===================================")

y_val_pred = classifier.predict(
    X_val_scaled
)

y_val_proba = classifier.predict_proba(
    X_val_scaled
)

positive_idx = list(
    classifier.classes_
).index(1)

y_val_proba = y_val_proba[:, positive_idx]

val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

val_precision = precision_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_recall = recall_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_f1 = f1_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_ap = average_precision_score(
    y_val,
    y_val_proba
)

val_auc = roc_auc_score(
    y_val,
    y_val_proba
)

print(f"Accuracy:          {val_accuracy:.4f}")
print(f"Precision:         {val_precision:.4f}")
print(f"Recall:            {val_recall:.4f}")
print(f"F1 Score:          {val_f1:.4f}")
print(f"Average Precision: {val_ap:.4f}")
print(f"ROC-AUC:           {val_auc:.4f}")



Training TabPFN...

TabPFN training completed!

VALIDATION RESULTS (2023)


In [ ]:
# ------------------------------------------------------------
# 4. Confusion matrix
# ------------------------------------------------------------

print("\n-----------------------------------")
print("CONFUSION MATRIX")
print("-----------------------------------")

val_cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=[0, 1]
)

print(val_cm)

print("\nRows = Actual classes")
print("Columns = Predicted classes")
print("Class 0 = No detected fire")
print("Class 1 = Detected fire")


# ------------------------------------------------------------
# 5. Classification report
# ------------------------------------------------------------

print("\n-----------------------------------")
print("CLASSIFICATION REPORT")
print("-----------------------------------")

print(
    classification_report(
        y_val,
        y_val_pred,
        labels=[0, 1],
        target_names=[
            "No detected fire",
            "Detected fire"
        ],
        zero_division=0
    )
)


# ------------------------------------------------------------
# 6. Save validation predictions
# ------------------------------------------------------------

import pandas as pd

validation_results = pd.DataFrame({
    "actual": np.asarray(y_val),
    "predicted": np.asarray(y_val_pred),
    "fire_probability": y_val_proba
})

validation_results.to_csv(
    "/content/validation_predictions_2023.csv",
    index=False
)

print("\nValidation predictions saved successfully.")

print("\n===================================")
print("VALIDATION EVALUATION COMPLETED")
print("===================================")

In [ ]:
# ============================================================
# CHECK FIRMS AND MERGED FIRMS + ERA5 DATASET SIZES
# ============================================================

import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/datasets_gaw"

# File paths
firms_path = os.path.join(
    DATA_DIR,
    "firms_h3_cell_date.csv"
)

merged_path = os.path.join(
    DATA_DIR,
    "firms_era5_nearest_neighbor.csv"
)

# Load datasets
firms_df = pd.read_csv(firms_path)

merged_df = pd.read_csv(merged_path)

# Display dataset information
datasets = {
    "FIRMS H3 Dataset": firms_df,
    "Merged FIRMS + ERA5 Dataset": merged_df
}

for name, df in datasets.items():

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    print(f"Number of rows:       {len(df):,}")
    print(f"Number of columns:    {len(df.columns):,}")

    if "h3_cell" in df.columns:
        print(
            f"Unique H3 cells:      "
            f"{df['h3_cell'].nunique():,}"
        )

    date_col = (
        "acq_date" if "acq_date" in df.columns
        else "valid_time" if "valid_time" in df.columns
        else None
    )

    if date_col:
        dates = pd.to_datetime(
            df[date_col],
            errors="coerce"
        )

        print("Date range:", dates.min(), "to", dates.max())

    print("\nColumns:")
    print(df.columns.tolist())


# ============================================================
# COMPARE DATASET SIZES
# ============================================================

comparison = pd.DataFrame({
    "Dataset": [
        "FIRMS H3",
        "Merged FIRMS + ERA5"
    ],
    "Rows": [
        len(firms_df),
        len(merged_df)
    ],
    "Columns": [
        len(firms_df.columns),
        len(merged_df.columns)
    ],
    "Unique H3 cells": [
        firms_df["h3_cell"].nunique(),
        merged_df["h3_cell"].nunique()
    ]
})

print("\n\nDATASET SIZE COMPARISON")
display(comparison)